# LangGraph 심화 실습

04_langgraph_basics, 01_graph_api, 02_workflows에서 배운 내용을 복습하는 실습입니다.

`___` 부분을 채워 넣으세요.

> practice_exercise와 겹치지 않도록, 리듀서 / 조건부 엣지 / MessagesState / 입출력 스키마 / 5대 워크플로 패턴을 다룹니다.

## 0. 환경 설정

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-5.4")
print("\u2713 \ubaa8\ub378 \uc900\ube44 \uc644\ub8cc")

---
## 1단계: 상태 리듀서 (01_graph_api 복습)

리듀서 없이 상태를 업데이트하면 **덮어쓰기**가 됩니다.
`Annotated`와 `operator.add`를 사용하면 리스트에 **누적**할 수 있습니다.

아래 코드에서 `items`는 누적되고, `count`는 덮어쓰기되도록 State를 정의하세요.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import ___, TypedDict   # Q1. Annotated를 import
import ___                           # Q2. operator 모듈 import

# Q3. items는 리스트 누적(operator.add), count는 단순 덮어쓰기
class ________(TypedDict):
    items: ___[list[str], ___.___]   # 리듀서: 리스트 누적
    count: int                        # 리듀서 없음: 덮어쓰기

def step_login(state: LogState) -> dict:
    return {"items": ["login"], "count": 1}

def step_search(state: LogState) -> dict:
    return {"items": ["search"], "count": 2}

def step_logout(state: LogState) -> dict:
    return {"items": ["logout"], "count": 3}

builder = ________(LogState)
builder.________("login", step_login)
builder.________("search", step_search)
builder.________("logout", step_logout)

builder.________(________, "login")
builder.________("login", "search")
builder.________("search", "logout")
builder.________("logout", ________)

graph = builder.________()
result = graph.in________voke({"items": [], "count": 0})

# items는 누적: ['login', 'search', 'logout'], count는 마지막 값: 3
print(f"items: {result['items']}")  # ['login', 'search', 'logout']
print(f"count: {result['count']}")  # 3

---
## 2단계: 조건부 엣지 (01_graph_api 복습)

상태 값에 따라 **다른 노드로 분기**하는 그래프를 만듭니다.

숫자가 양수면 `"positive"` 노드로, 음수면 `"negative"` 노드로, 0이면 `"zero"` 노드로 이동합니다.

```
START → check → [route] → positive → END
                        → negative → END
                        → zero     → END
```

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class NumberState(TypedDict):
    number: int
    label: str
    result: str

def check_number(state: NumberState) -> dict:
    n = state["number"]
    if n > 0:
        return {"label": "positive"}
    elif n < 0:
        return {"label": "negative"}
    return {"label": "zero"}

def handle_positive(state: NumberState) -> dict:
    return {"result": f"{state['number']}\uc740 \uc591\uc218\uc785\ub2c8\ub2e4."}

def handle_negative(state: NumberState) -> dict:
    return {"result": f"{state['number']}\uc740 \uc74c\uc218\uc785\ub2c8\ub2e4."}

def handle_zero(state: NumberState) -> dict:
    return {"result": "0\uc785\ub2c8\ub2e4."}

# Q4. 라우팅 함수: state의 label 값을 반환
def route_number(state: NumberState) -> str:
    return state["label"]  # "positive", "negative", "zero" 중 하나

builder = StateGraph(NumberState)
# Input your code
# Input your code
# Input your code
# Input your code

builder.add_edge(START, "check")

# Q5. add_conditional_edges: 소스 노드, 라우팅 함수, 매핑 딕셔너리
builder._______________("check", route_number, {
    "positive": "positive",
    "negative": "negative",
    "zero": "zero",
})

# Q6. 각 처리 노드 → END로 연결
builder.________("positive", END)
builder.________("negative", END)
builder.________("zero", END)

graph = builder.compile()

for num in [42, -7, 0]:
    result = graph.invoke({"number": num})
    print(result["result"])

---
## 3단계: MessagesState + LLM 노드 (04_langgraph_basics 복습)

`MessagesState`를 사용하면 LLM 대화를 그래프로 구성할 수 있습니다.
MessagesState 안에는 `messages: Annotated[list, add_messages]`가 이미 정의되어 있어서,
메시지를 반환하면 자동으로 **누적**됩니다.

In [ ]:
from langgraph.graph import ___, START, END  # Q7. MessagesState를 import
from langchain.messages import HumanMessage, ___  # Q8. SystemMessage를 import

# Q9. chatbot 노드: state의 messages를 model.invoke()에 전달하고,
#     응답을 messages 리스트에 넣어 반환
def chatbot(state: MessagesState) -> dict:
    response = model.___(state["___"])
    return {"messages": [___]}

builder = StateGraph(___)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = ________.________()

# Q10. SystemMessage + HumanMessage로 호출
result = ________.________({
    "messages": [
        ___________(content="\ub2f9\uc2e0\uc740 \uc218\ud559 \uc120\uc0dd\ub2d8\uc785\ub2c8\ub2e4."),
        ___________(content="\ud53c\ud0c0\uace0\ub77c\uc2a4 \uc815\ub9ac\ub97c \ud55c \ubb38\uc7a5\uc73c\ub85c \uc124\uba85\ud574\uc8fc\uc138\uc694."),
    ]
})

print("\uc751\ub2f5:", result["messages"][-1].content)

---
## 4단계: 입출력 스키마 (01_graph_api 복습)

내부 상태에는 중간 처리용 필드(`intermediate`)가 있지만,
외부에는 `question`만 받고 `answer`만 내보내도록 입출력 스키마를 분리합니다.

```
외부 입력(question) → 내부 처리(intermediate) → 외부 출력(answer)
```

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Q11. 입력 스키마: question만 받음
class InputSchema(TypedDict):
    ___: str

# Q12. 출력 스키마: answer만 내보냄
class OutputSchema(TypedDict):
    ___: str

# 내부 상태: 중간 처리 필드 포함
class InternalState(TypedDict):
    question: str
    answer: str
    intermediate: str  # 외부에 노출되지 않음

def preprocess(state: InternalState) -> dict:
    return {"intermediate": state["question"].upper()}

def generate_answer(state: InternalState) -> dict:
    return {"answer": f"'{state['intermediate']}'\uc5d0 \ub300\ud55c \ub2f5\ubcc0\uc785\ub2c8\ub2e4."}

# Q13. StateGraph에 내부 상태, 입력 스키마, 출력 스키마를 전달
builder = StateGraph(
    ___,
    input_schema=___,
    output_schema=___,
)

builder.________("preprocess", preprocess)
builder.________("answer", generate_answer)

builder.________(________, "preprocess")
builder.________("preprocess", "answer")
builder.________("answer", ________)

graph = builder.compile()

# 입력은 question만, 출력은 answer만 나옴 (intermediate는 숨겨짐)
result = ________.________({"question": "\ub7ad\uadf8\ub798\ud504\ub780?"})
print(result)  # {'answer': "'랭그래프란?'에 대한 답변입니다."}

---
## 5단계: Prompt Chaining 패턴 (02_workflows 복습)

첫 번째 LLM이 **초안**을 쓰고, 두 번째 LLM이 **개선**하는 순차 체인입니다.

```
START → draft → improve → END
```

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class ChainState(TypedDict):
    topic: str
    draft: str
    improved: str

# Q14. topic을 받아 한 문장 설명을 생성하는 노드
def generate_draft(state: ChainState) -> dict:
    response = model.___(f"\ub2e4\uc74c\uc5d0 \ub300\ud574 \ud55c \ubb38\uc7a5\uc73c\ub85c \uc124\uba85\ud574\uc8fc\uc138\uc694: {state['___']}")
    return {"draft": response.___}

# Q15. draft를 받아 더 좋게 다듬는 노드
def improve_draft(state: ChainState) -> dict:
    response = model.___(f"\ub2e4\uc74c \ubb38\uc7a5\uc744 \ub354 \ub9e4\ub825\uc801\uc73c\ub85c \uac1c\uc120\ud574\uc8fc\uc138\uc694: {state['___']}")
    return {"improved": response.___}

builder = StateGraph(ChainState)

# Q16. 노드 등록 및 엣지 연결
________.________("___", generate_draft)
________.________("___", improve_draft)

________.________(___, "draft")
________.________("draft", "___")
________.________("___", END)

chain = ________.________()
result = ________.________({"topic": "\uc778\uacf5\uc9c0\ub2a5"})
print(f"\ucd08\uc548: {result['draft']}")
print(f"\uac1c\uc120: {result['improved']}")

---
## 6단계: Parallelization 패턴 (02_workflows 복습)

하나의 텍스트에 대해 **감정 분석**과 **키워드 추출**을 **동시에** 수행하고, 결과를 합칩니다.

```
        ┌→ sentiment ─┐
START ──┤              ├→ synthesize → END
        └→ keywords  ─┘
```

**핵심**: `analyses`에 `Annotated[list[str], operator.add]` 리듀서를 써서 두 노드의 결과를 누적합니다.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import Annotated, TypedDict
import operator

# Q17. analyses를 리스트 누적 리듀서로 정의
class ParallelState(TypedDict):
    text: str
    analyses: ___[list[str], ___.___]

def analyze_sentiment(state: ParallelState) -> dict:
    r = model.invoke(f"\ud55c \ubb38\uc7a5\uc73c\ub85c \uac10\uc815\uc744 \ubd84\uc11d\ud574\uc8fc\uc138\uc694: {state['text']}")
    return {"analyses": [f"\uac10\uc815: {r.content}"]}

def extract_keywords(state: ParallelState) -> dict:
    r = model.invoke(f"\ud575\uc2ec \ud0a4\uc6cc\ub4dc 3\uac1c\ub97c \ucd94\ucd9c\ud574\uc8fc\uc138\uc694: {state['text']}")
    return {"analyses": [f"\ud0a4\uc6cc\ub4dc: {r.content}"]}

def synthesize(state: ParallelState) -> dict:
    return {"analyses": [f"\uc885\ud569: {len(state['analyses'])}\uac1c \ubd84\uc11d \uc644\ub8cc"]}

builder = StateGraph(ParallelState)
builder.add_node("________", ________)
builder.add_node("________", ________)
builder.add_node("________", synthesize)

# Q18. START에서 두 분석 노드로 병렬 분기
builder.add_edge(___, "sentiment")
builder.add_edge(___, "keywords")

# Q19. 두 노드 완료 후 synthesize로 합류
builder.add_edge("___", "synthesize")
builder.add_edge("___", "synthesize")

builder.add_edge("synthesize", ___)

parallel_graph = builder.compile()
result = parallel_graph.invoke({"text": "AI \uae30\uc220\uc774 \uc758\ub8cc \ubd84\uc57c\ub97c \ud601\uc2e0\ud558\uace0 \uc788\ub2e4.", "analyses": []})

for a in result["analyses"]:
    print(f"  {a}")

---
## 7단계: Routing 패턴 (02_workflows 복습)

LLM의 **structured output**으로 질문을 분류하고, 카테고리별 전문가가 답변합니다.

```
START → classify → [route] → science → END
                            → history → END
                            → culture → END
```

In [ ]:
from pydantic import BaseModel
from typing import Literal, TypedDict
from langgraph.graph import StateGraph, START, END

# Q20. Pydantic 모델로 분류 결과를 정의 (science / history / culture)
class Classification(BaseModel):
    category: Literal["___", "___", "___"]

class RouteState(TypedDict):
    question: str
    category: str
    answer: str

# Q21. LLM의 structured output으로 분류
def classify(state: RouteState) -> dict:
    structured = model.___(Classification)
    result = structured.________(f"\ub2e4\uc74c \uc9c8\ubb38\uc744 science/history/culture \uc911 \ud558\ub098\ub85c \ubd84\ub958\ud558\uc138\uc694: {state['question']}")
    return {"category": result.___}

def handle_science(state: RouteState) -> dict:
    r = model.________(f"\uacfc\ud559 \uc804\ubb38\uac00\ub85c\uc11c \ub2f5\ubcc0: {state['question']}")
    return {"answer": r.content}

def handle_history(state: RouteState) -> dict:
    r = model.________(f"\uc5ed\uc0ac \uc804\ubb38\uac00\ub85c\uc11c \ub2f5\ubcc0: {state['question']}")
    return {"answer": r.content}

def handle_culture(state: RouteState) -> dict:
    r = model.________(f"\ubb38\ud654 \uc804\ubb38\uac00\ub85c\uc11c \ub2f5\ubcc0: {state['question']}")
    return {"answer": r.content}

# Q22. 라우팅 함수
def route(state: RouteState) -> str:
    return state["___"]

builder = StateGraph(RouteState)
builder.________("classify", classify)
builder.________("science", handle_science)
builder.________("history", handle_history)
builder.________("culture", handle_culture)

builder.________(START, "classify")

# Q23. 조건부 엣지: classify → route 함수로 분기
builder.___(
    "classify",
    ___,
    {
        "science": "science",
        "history": "history",
        "culture": "culture",
    }
)

builder.________("science", END)
builder.________("history", END)
builder.________("culture", END)

router = builder.compile()

result = router.invoke({"question": "\ube57\uc758 \uc18d\ub3c4\ub294 \uc5bc\ub9c8\uc778\uac00\uc694?"})
print(f"\uce74\ud14c\uace0\ub9ac: {result['category']}")
print(f"\ub2f5\ubcc0: {result['answer'][:200]}")

---
## 8단계: Orchestrator-Worker 패턴 (02_workflows 복습)

오케스트레이터가 **섹션 계획**을 세우고, `Send()`로 **워커를 동적으로 생성**하여 각 섹션을 작성합니다.

```
START → plan → [Send] → worker(섹션1) ─┐
                      → worker(섹션2) ─┼→ END
                      → worker(섹션3) ─┘
```

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import ___  # Q24. Send를 import
from typing import Annotated, TypedDict
import operator

class OrchestratorState(TypedDict):
    topic: str
    sections: list[str]
    results: Annotated[list[str], operator.add]

# 워커가 받는 상태 (섹션 1개)
class WorkerState(TypedDict):
    section: str

# 오케스트레이터: 섹션 3개를 계획
def plan_sections(state: OrchestratorState) -> dict:
    r = model.invoke(
        f"'{state['topic']}'\uc5d0 \ub300\ud55c \uc9e7\uc740 \uc139\uc158 \uc81c\ubaa9 3\uac1c\ub97c \ub098\uc5f4\ud574\uc8fc\uc138\uc694. \ud55c \uc904\uc5d0 \ud558\ub098\uc529, \ubc88\ud638 \uc5c6\uc774."
    )
    sections = [s.strip() for s in r.content.strip().split("\n") if s.strip()][:3]
    return {"sections": sections}

# Q25. Send()로 각 섹션을 worker 노드에 분배
def assign_workers(state: OrchestratorState) -> list[Send]:
    return [
        ___("worker", {"___": s})
        for s in state["sections"]
    ]

# 워커: 섹션 1개에 대해 한 문장 작성
def worker(state: WorkerState) -> dict:
    r = model.invoke(f"\ub2e4\uc74c\uc5d0 \ub300\ud574 \ud55c \ubb38\uc7a5\uc73c\ub85c \uc791\uc131\ud574\uc8fc\uc138\uc694: {state['section']}")
    return {"results": [f"## {state['section']}\n{r.content}"]}

builder = ________(OrchestratorState)
builder.________("plan", plan_sections)
builder.________("worker", worker)

builder.________(START, "plan")

# Q26. plan 노드에서 assign_workers 함수로 조건부 엣지
builder.___("plan", ___, ["worker"])

builder.add_edge("worker", END)

orchestrator = builder.________()
result = ________.________({"topic": "\ud074\ub77c\uc6b0\ub4dc \ucef4\ud4e8\ud305", "sections": [], "results": []})

for r in result["results"]:
    print(r)
    print()

---
## 9단계: Evaluator-Optimizer 패턴 (02_workflows 복습)

생성 → 평가 → (점수가 낮으면) 다시 생성하는 **반복 루프**입니다.
8점 이상이거나 3회 반복하면 종료합니다.

```
START → generate → evaluate → [should_retry] → generate (반복)
                                              → END (종료)
```

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class EvalState(TypedDict):
    task: str
    draft: str
    feedback: str
    is_good: bool
    iterations: int

# Q27. 생성 노드: 피드백이 있으면 개선, 없으면 새로 생성
def generate(state: ________) -> dict:
    if state.get("___"):  # 피드백이 있으면
        prompt = f"\ud53c\ub4dc\ubc31\uc744 \ubc18\uc601\ud558\uc5ec \uac1c\uc120\ud574\uc8fc\uc138\uc694.\n\uc6d0\ubcf8: {state['draft']}\n\ud53c\ub4dc\ubc31: {state['feedback']}"
    else:
        prompt = f"\ub2e4\uc74c\uc5d0 \ub300\ud55c \ud55c \ubb38\uc7a5 \uc2ac\ub85c\uac74\uc744 \uc791\uc131\ud574\uc8fc\uc138\uc694: {state['task']}"
    r = model.invoke(prompt)
    return {"draft": r.content, "iterations": state.get("iterations", 0) + 1}

# Q28. 평가 노드: 1~10 점수를 매기고, 8점 이상이면 is_good=True
def evaluate(state: ________) -> dict:
    r = model.________(f"\uc774 \uc2ac\ub85c\uac74\uc744 1-10\uc73c\ub85c \ud3c9\uac00\ud558\uace0 \uac04\ub2e8\ud55c \ud53c\ub4dc\ubc31\uc744 \uc8fc\uc138\uc694: '{state['draft']}'")
    content = r.content
    is_good = any(f"{n}/10" in content for n in range(___, ___))  # 8, 9, 10
    return {"feedback": content, "is_good": is_good}

# Q29. 재시도 판단: is_good이거나 3회 이상이면 END, 아니면 "generate"
def should_retry(state: ________) -> str:
    if state["___"] or state["___"] >= 3:
        return ___
    return "___"

builder = ________(EvalState)
builder.________("generate", ________)
builder.________("evaluate", ________)

builder.________(START, "generate")
builder.________("generate", "evaluate")

# Q30. evaluate에서 should_retry로 조건부 엣지 (generate로 돌아가거나 END)
builder.___("evaluate", ___, ["generate", END])

optimizer = builder.compile()
result = optimizer.invoke({"task": "\ud658\uacbd \ubcf4\ud638 \ucea0\ud398\uc778"})
print(f"\ucd5c\uc885 \uc2ac\ub85c\uac74 ({result['iterations']}\ubc88 \ubc18\ubcf5): {result['draft']}")

---
## 수고하셨습니다!

정답은 `langgraph_exercise_answer.ipynb`에서 확인하세요.
